# 02 - How Modern RAG Works

## Definition
Modern RAG is a staged pipeline that separates retrieval quality from generation quality.

## Theory
Dense embeddings map semantics to vectors; vector search returns nearest contexts; prompt assembly grounds response synthesis.

## Motivation
Debugging is easier when each stage has independent metrics and diagnostics.

## Architecture
User Query -> Embedding -> Vector Search -> Retrieved Context -> Prompt Construction -> Generation -> Answer.

## Real-world Examples
- Enterprise support assistant over product docs
- Compliance assistant over policy text
- Research assistant over paper abstracts

## Best Practices
- Measure retrieval and generation separately
- Track latency and grounding together
- Keep prompts strict about citations and uncertainty

## Common Mistakes
- Skipping retrieval diagnostics
- Using mismatched embedding models for index/query
- Reporting only one metric and claiming broad quality


In [ ]:
from pathlib import Path
import json
import pandas as pd

from rag_system import (
    RAGConfig,
    ProjectRunner,
    ChunkingStrategy,
)

cfg = RAGConfig(project_root=Path(".."), profile="balanced")
runner = ProjectRunner(cfg)


In [ ]:
bundle = runner.ingest_dataset()
len(bundle['documents']), len(bundle['queries']), bundle['summary']


In [ ]:
chunking = runner.run_chunking(bundle['documents'][:260], strategy=ChunkingStrategy.RECURSIVE)
runner.index_chunks(chunking.chunks, reset=True)

sample_query = bundle['queries'][0].query
result = runner.pipeline.answer(sample_query, top_k=6)

{
    'query': sample_query,
    'answer_preview': result.answer[:280],
    'abstained': result.abstained,
    'num_retrieved': len(result.retrieved_chunks),
    'citations': result.citations,
}


## Code Explanation
This walkthrough runs a minimal retrieve-augment-generate loop and inspects retrieved chunks before trusting output text.